# Cheat Sheet: Preprocessing Pipelines with scikit-learn
### Quick reference & runnable snippets for the Penguin Classification Project

Run the setup cell first, then jump to whichever section you need.

In [ ]:
# --- Setup ---
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, OrdinalEncoder
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('https://codefinity-content-media.s3.eu-west-1.amazonaws.com/a65bbc96-309e-4df9-a790-a1eb8c815a1c/penguins.csv')
df = df[df.isna().sum(axis=1) < 2]
X, y = df.drop('species', axis=1), df['species']
df.head()

## 1. `OneHotEncoder` — syntax variants

In [ ]:
# Basic usage (fit_transform on a DataFrame subset)
ohe = OneHotEncoder()
encoded = ohe.fit_transform(X[['sex', 'island']])   # returns a SPARSE matrix by default
print(type(encoded))

# Dense output (easier to inspect, use for small/medium data)
ohe_dense = OneHotEncoder(sparse_output=False)      # sklearn >=1.2 arg name; older: sparse=False
dense = ohe_dense.fit_transform(X[['sex', 'island']])
print(dense[:5])

# See the generated column names
print(ohe_dense.get_feature_names_out(['sex', 'island']))

# Handle unseen categories at predict time gracefully instead of erroring
ohe_safe = OneHotEncoder(handle_unknown='ignore')

# Drop first category per column to avoid perfect multicollinearity (useful for linear models)
ohe_drop = OneHotEncoder(drop='first')

## 2. `SimpleImputer` — strategies

In [ ]:
# Numeric column, fill with mean
imp_mean = SimpleImputer(strategy='mean')

# Numeric column, robust to outliers
imp_median = SimpleImputer(strategy='median')

# Works on numeric OR categorical/object columns — fills with the mode
imp_mode = SimpleImputer(strategy='most_frequent')

# Fill with an explicit constant (e.g. flag missing category)
imp_const = SimpleImputer(strategy='constant', fill_value='Unknown')

# Example
sample = pd.DataFrame({'a': [1, np.nan, 3]})
print(SimpleImputer(strategy='mean').fit_transform(sample))

## 3. `StandardScaler` / alternatives

In [ ]:
# Zero mean, unit variance (the default choice for most models)
scaler = StandardScaler()

# Squash into [0, 1] range instead — useful for neural nets / bounded activations
minmax = MinMaxScaler()

sample = pd.DataFrame({'x': [10, 20, 30, 40]})
print('StandardScaler:', StandardScaler().fit_transform(sample).ravel())
print('MinMaxScaler:  ', MinMaxScaler().fit_transform(sample).ravel())

# Inspect learned statistics after fitting
s = StandardScaler().fit(sample)
print('mean_:', s.mean_, ' scale_ (std):', s.scale_)

## 4. `ColumnTransformer` — shorthand vs. explicit

In [ ]:
# Shorthand: make_column_transformer — auto-names each transformer
ct_short = make_column_transformer(
    (OneHotEncoder(), ['sex', 'island']),
    remainder='passthrough'
)

# Explicit: ColumnTransformer — you choose the names (useful for GridSearchCV param keys)
ct_explicit = ColumnTransformer(
    transformers=[
        ('cat_encoder', OneHotEncoder(), ['sex', 'island']),
        ('num_scaler', StandardScaler(), ['bill_length_mm', 'bill_depth_mm'])
    ],
    remainder='passthrough'   # default is 'drop' -- easy to forget!
)

# remainder options:
#   'drop'        -> unlisted columns are REMOVED (default, often a silent bug source)
#   'passthrough' -> unlisted columns pass through unchanged
#   an estimator  -> apply that transformer to all remaining columns

print(ct_short)

## 5. `Pipeline` — shorthand vs. explicit

In [ ]:
# Shorthand: make_pipeline — step names auto-generated (lowercase class name)
pipe_short = make_pipeline(
    make_column_transformer((OneHotEncoder(), ['sex', 'island']), remainder='passthrough'),
    SimpleImputer(strategy='most_frequent'),
    StandardScaler()
)
print(list(pipe_short.named_steps.keys()))

# Explicit: Pipeline — you choose step names
pipe_explicit = Pipeline(steps=[
    ('preprocess', make_column_transformer((OneHotEncoder(), ['sex', 'island']), remainder='passthrough')),
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('scale', StandardScaler())
])
print(list(pipe_explicit.named_steps.keys()))

## 6. Fitting, transforming, and inspecting a pipeline

In [ ]:
fitted = pipe_short.fit(X)
X_transformed = fitted.transform(X)          # or pipe_short.fit_transform(X)
print(X_transformed.shape)
print(X_transformed[:3])

# Access an individual step after fitting
ct_step = fitted.named_steps['columntransformer']
print(ct_step)

# Get output feature names all the way through the ColumnTransformer step
print(ct_step.get_feature_names_out())

## 7. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% held out for testing
    random_state=42,      # reproducibility
    stratify=y            # keep class proportions the same in train & test
)
print(X_train.shape, X_test.shape)

## 8. Full pipeline WITH a classifier at the end

In [ ]:
full_pipe = make_pipeline(
    make_column_transformer((OneHotEncoder(handle_unknown='ignore'), ['sex', 'island']),
                             remainder='passthrough'),
    SimpleImputer(strategy='most_frequent'),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

full_pipe.fit(X_train, y_train)
preds = full_pipe.predict(X_test)
print('Accuracy:', accuracy_score(y_test, preds))
print(classification_report(y_test, preds))
print(confusion_matrix(y_test, preds))

## 9. Cross-validation

In [ ]:
scores = cross_val_score(full_pipe, X, y, cv=5, scoring='accuracy')
print('Fold scores:', scores)
print('Mean +/- std:', scores.mean(), '+/-', scores.std())

## 10. Hyperparameter tuning with `GridSearchCV` on a pipeline

Note the `stepname__paramname` syntax to reach into pipeline steps.

In [ ]:
param_grid = {
    'kneighborsclassifier__n_neighbors': [3, 5, 7, 9],
    'kneighborsclassifier__weights': ['uniform', 'distance']
}

grid = GridSearchCV(full_pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X, y)
print('Best params:', grid.best_params_)
print('Best CV score:', grid.best_score_)

## 11. Common Pitfalls

| Pitfall | Why it hurts | Fix |
|---|---|---|
| Forgetting `remainder='passthrough'` | `ColumnTransformer` silently **drops** every column not explicitly listed | Always set `remainder` deliberately |
| Fitting the scaler/imputer on the whole dataset before splitting | Leaks test-set statistics into training → overly optimistic scores | Fit only on `X_train` (a `Pipeline` + `cross_val_score`/`GridSearchCV` does this automatically) |
| Using `OneHotEncoder` without `handle_unknown='ignore'` in production | Crashes if a new category appears at predict time | Set `handle_unknown='ignore'` |
| Comparing KNN/SVM performance without scaling | Large-magnitude features dominate distance calculations | Always scale numeric features for distance-based models |
| Using `strategy='mean'` on a column that's actually categorical | Produces meaningless fractional "categories" | Use `'most_frequent'` (or a separate imputer per column type) |
| Calling `.fit_transform()` on test data | Refits statistics on test data instead of reusing training statistics | Use `.transform()` (not `.fit_transform()`) on test/validation data |
